In [1]:
import pandas as pd
import polars as pl
import plotly_express as px
from utils import *

In [2]:
data = last_full_pull()

Loading most recent full pull file...
Loading file: ../Data/Full Pulls\ygo_full_daily_2025-10-14_19-22-08.csv


In [3]:
data.head(5)

,card_id,name,desc,pend_desc,monster_desc,type,subtype,frame,race,archetype,...,min_price,price_range,price_stddev,is_tcg_exclusive,is_ocg_exclusive,days_since_tcg_release,days_since_ocg_release,first_market,market_delay,desc_length
0,80181649,"""A Case for K9""","When this card is activated: You can add 1 ""K9...",NaN,NaN,Spell Card,Continuous Spell,spell,Continuous,K9,...,0.14,0.00,NaN,0,0,75.0,206.0,OCG,131.0,92
1,80181649,"""A Case for K9""","When this card is activated: You can add 1 ""K9...",NaN,NaN,Spell Card,Continuous Spell,spell,Continuous,K9,...,0.14,0.00,NaN,0,0,75.0,206.0,OCG,131.0,92
2,34541863,"""A"" Cell Breeding Device","During each of your Standby Phases, put 1 A-Co...",NaN,NaN,Spell Card,Continuous Spell,spell,Continuous,Alien,...,0.11,24.34,12.006210,0,0,6726.0,6816.0,OCG,90.0,16
3,64163367,"""A"" Cell Incubator",Each time an A-Counter(s) is removed from play...,NaN,NaN,Spell Card,Continuous Spell,spell,Continuous,Alien,...,0.25,1.00,0.473242,0,0,6544.0,6660.0,OCG,116.0,32
4,91231901,"""A"" Cell Recombination Device",Target 1 face-up monster on the field; send 1 ...,NaN,NaN,Spell Card,Quick-Play Spell,spell,Quick-Play,Alien,...,0.25,0.74,0.310631,0,0,3267.0,3384.0,OCG,117.0,66


In [4]:
print("Number of unique *printings*:")
print(data.shape[0])
print("Number of unique *cards*:")
print(data['card_id'].nunique())

Number of unique *printings*:
39786
Number of unique *cards*:
13481


In [5]:
# Write code to filter data to the oldest tcg_date for each card_id
data_all = data.copy()
data_all['tcg_date_dt'] = pd.to_datetime(data_all['tcg_date'], errors='coerce')
oldest_dates = data_all.groupby('card_id')['tcg_date_dt'].min().reset_index()
oldest_dates.columns = ['card_id', 'oldest_tcg_date']
data_all = data_all.merge(oldest_dates, on='card_id')
data = data_all[data_all['tcg_date_dt'] == data_all['oldest_tcg_date']]
data = data.drop(columns=['oldest_tcg_date'])
# Now for any card_id with multiple entries (e.g. multiple printings on the same day), keep only the first one
data = data.drop_duplicates(subset=['card_id'], keep='first')
data.shape

(13480, 57)

In [6]:
FRAME_COLORS = {
    "Spell Card": "#228B22",                    # Forest Green

    # Monster categories grouped by type similarity and common frame colors
    "Normal Monster": "#EBDF9E",                # Beige/cream - Normal monsters
    "Effect Monster": "#D2691E",                # Brown-orange
    "Flip Effect Monster": "#D97A25",           # Slightly lighter brown-orange, related to Effect Monster
    "Union Effect Monster": "#C2691E",          # Related brown-orange shade
    "Pendulum Effect Monster": "#CC7A32",       # Related to Effect Monster, with distinct slightly lighter tone
    "Pendulum Effect Fusion Monster": "#803080", # Purplish, fusion related
    "Pendulum Normal Monster": "#E9DCC9",       # Very light beige, near normal
    "Synchro Monster": "#FFFFFF",                # White
    "Synchro Tuner Monster": "#E6E6E6",          # Near white, related to Synchro Monster
    "Synchro Pendulum Effect Monster": "#F0F0F0", # Slightly off white, pendulum synchro
    "Tuner Monster": "#CCCCCC",                   # Gray, related to Synchro Tuner
    "Normal Tuner Monster": "#DDDDDD",            # Light gray, normal tuner
    "Gemini Monster": "#D0C5A2",                   # Slightly darker beige, related to Normal/Effect
    "Spirit Monster": "#D0B983",                   # Light goldish, related but distinct
    "Ritual Effect Monster": "#4169E1",            # Royal blue, ritual
    "Ritual Monster": "#3A5FCD",                   # Slightly darker blue, related ritual
    "Toon Monster": "#FFB6C1",                      # Light Pink, cartoonish theme

    # Trap Card
    "Trap Card": "#C71585",                      # Medium Violet Red (pinkish purple)

    # Other monster special frames
    "Link Monster": "#1E90FF",                    # Dodger Blue
    "XYZ Monster": "#000000",                     # Black
    "XYZ Pendulum Effect Monster": "#222222",     # Dark gray, similar to XYZ but distinct
    "Pendulum Effect Ritual Monster": "#3C50B4", # Related to Ritual, pendulum effect
    "Pendulum Flip Effect Monster": "#DB8B3A",    # Orangish, related to pendulum effect
    "Flip Tuner Effect Monster": "#B97416",       # Brownish, closer to tuner effect

    # Less common or specialized cards
    "Skill Card": "#6B8E23",                      # Olive green, distinct from spell cards
    "Token": "#808080",                           # Gray
}


In [7]:
type_counts = data['type'].value_counts().reset_index()
type_counts.columns = ['type', 'count']
fig = px.bar(
    type_counts, 
    x='count',
    y='type', 
    category_orders={'type': type_counts.sort_values('count', ascending=False)['type'].tolist()},
    title="Card Type Distribution",
    text='count',
    color='type',
    color_discrete_map=FRAME_COLORS,
    height=800,
    template='ggplot2'
)
fig.update_yaxes(showgrid=False)
fig.show()

In [8]:
data['tcg_date_dt'] = pd.to_datetime(data['tcg_date'], errors='coerce')
data['release_year'] = data['tcg_date_dt'].dt.year
yearly_counts = data[data['release_year'] < 2025].groupby(['type', 'release_year']).size().reset_index(name='count')
fig = px.line(
    yearly_counts,
    x='release_year',
    y='count',
    color='type',
    color_discrete_map=FRAME_COLORS,
    title='Yearly Card Releases by Type',
    markers=True,
    height=850,
    template='ggplot2'
)
fig.update_xaxes(showgrid=False)
# Update the axis labels
fig.update_layout(xaxis_title='Release Year', yaxis_title='Number of Cards')
# Update legend title
fig.update_layout(legend_title_text='Card Type')
fig.show()

In [9]:
archetype_counts = data[data['archetype'] != ''].groupby('archetype').size().reset_index(name='len')
archetype_counts = archetype_counts.sort_values(by='len', ascending=False)

fig = px.bar(
    archetype_counts.head(20),
    x='len',
    y='archetype',
    category_orders={'archetype': archetype_counts.sort_values('len', ascending=False)['archetype'].tolist()},
    title='Top 20 Archetypes by Number of Unique Cards',
    text='len',
    height=600,
    template='ggplot2'
)

fig.update_yaxes(showgrid=False)
fig.update_layout(xaxis_title='Number of Unique Cards', yaxis_title='Archetype')
fig.show()

In [10]:
# Get all the cards with non-missing atk and def and values >= 0
atk_def_data = data[(data['atk'].notnull()) & (data['def'].notnull()) & (data['atk'] >= 0) & (data['def'] >= 0)]

# Make a boxplot of atk and def together
atk_def_melted = atk_def_data.melt(value_vars=['atk', 'def'], var_name='stat', value_name='value')
fig = px.box(
    atk_def_melted,
    x='stat',
    y='value',
    title='Boxplot of Attack and Defense Points',
    height=500,
    template='ggplot2',
    color='stat',
    color_discrete_map={'atk': 'blue', 'def': 'green'}
)
fig.update_yaxes(showgrid=False)
fig.update_layout(xaxis_title='Stat', yaxis_title='Points')
fig.show()

In [11]:
# Plot atk and def points over Date
fig = px.scatter(
    atk_def_data,
    x='tcg_date_dt',
    y='atk',
    title='Attack Points Over Time',
    height=500,
    template='ggplot2',
    trendline='lowess',
    # Make the lowess line standout more
    trendline_color_override='blue',
    labels={'tcg_date_dt': 'Release Date', 'atk': 'Attack Points'}
)
fig.update_yaxes(showgrid=False)
fig.update_layout(xaxis_title='Release Date', yaxis_title='Attack Points')
fig.show()

In [12]:
fig = px.histogram(
    data, 
    x='desc_length',
    title='Distribution of Card Description Lengths',
    height=500,
    template='ggplot2',
    # nbins=50,  
    marginal='box'
)
fig.update_yaxes(showgrid=False)
fig.update_layout(xaxis_title='Description Length (characters)', yaxis_title='Number of Cards')
fig.show()

In [13]:
# Plot histograms of desc_length for each type on top of each other with an opacity of 0.5
fig = px.histogram(
    data, 
    x='desc_length',
    color='type',
    color_discrete_map=FRAME_COLORS,
    title='Card Description Lengths by Type',
    height=800,
    template='ggplot2',
    opacity=0.5,
    barmode='overlay'
)
fig.update_yaxes(showgrid=False)
fig.update_layout(xaxis_title='Description Length (characters)', yaxis_title='Number of Cards')
fig.show()

In [14]:
# plot card length over time
fig = px.scatter(
    data,
    x='tcg_date_dt',
    y='desc_length',
    title='Card Description Length Over Time',
    height=500,
    opacity=0.6,
    # Make the points lightblue
    color_discrete_sequence=['lightblue'],
    template='ggplot2',
    trendline='lowess',
    trendline_color_override='blue',
    labels={'name' : 'Card Name',
            'tcg_date_dt': 'Release Date', 
            'desc_length': 'Description Length (characters)'}
)
# Change the background to white and the gridlines to light gray
fig.update_yaxes(showgrid=True, gridcolor='lightgray')
fig.update_layout(plot_bgcolor='white')
fig.update_layout(xaxis_title='Release Date', yaxis_title='Description Length (characters)')
fig.show()

In [154]:
long_desc = data.sort_values('desc_length', ascending=False)[['name', 'desc_length']].head(20)

fig = px.bar(
    long_desc,
    x='desc_length',
    y='name',
    category_orders={'name': long_desc.sort_values('desc_length', ascending=False)['name'].tolist()},
    title='Top 20 Cards by with the Most Text',
    text='desc_length',
    height=1200,
    template='ggplot2'
)
fig.update_yaxes(showgrid=False)
fig.update_layout(xaxis_title='Description Length (characters)', yaxis_title='Card Name')